In [ ]:
# Colab 請選 2026.07；安裝到獨立環境，避免更改核心已載入的 NumPy。
from pathlib import Path
import subprocess
import sys
import tempfile

assert sys.version_info[:3] == (3, 12, 13), "請使用書籍或 Colab 2026.07 的 Python 3.12.13"
python = sys.executable
if "google.colab" in sys.modules:
    %pip -q install virtualenv
    env = Path("/content/moleculenet-book-env")
    if not (env / "bin/python").is_file():
        subprocess.run([python, "-m", "virtualenv", "-q", str(env)], check=True)
    python = str(env / "bin/python")
    pins = ["deepchem==2.8.1.dev20260821122258", "numpy==1.26.4", "scipy==1.17.1",
            "rdkit==2026.9.1", "scikit-learn==1.9.1", "pandas==3.0.6", "matplotlib==3.11.2"]
    subprocess.run([python, "-m", "pip", "install", "-q", *pins], check=True)
    subprocess.run([python, "-m", "pip", "check"], check=True)
work = tempfile.TemporaryDirectory()
script = Path(work.name) / "gaussian-processes.py"

# 水溶解度的預測範圍有多寬？

首次執行會下載實驗資料。可設定 DEEPCHEM_DATA_DIR 重用原始檔。要做練習，請在最後一格前新增以 `%%writefile -a "{script}"` 開頭的程式格。

In [ ]:
%%writefile "{script}"
from pathlib import Path
import os
import tempfile
import deepchem as dc
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
from rdkit import Chem
from sklearn.metrics import mean_absolute_error

SEED = 42
# ESOL: 1,128 筆水溶解度 log(mol/L)，DOI 10.1021/ci034243x。
cache = Path(os.environ.get("DEEPCHEM_DATA_DIR", Path.home() / ".cache" / "deepchem"))
cache.mkdir(parents=True, exist_ok=True)
figure_dir = Path("../../figures") if Path("../../outline.yml").is_file() else Path("figures")
figure_dir.mkdir(parents=True, exist_ok=True)

def read_splits(loader):
    # 原始檔可共用，每次建立的特徵快取則彼此獨立。
    with tempfile.TemporaryDirectory() as scratch:
        tasks, splits, _ = loader(
            featurizer=dc.feat.RawFeaturizer(smiles=True), splitter="scaffold",
            transformers=[], data_dir=str(cache), save_dir=scratch, reload=True)
        assert len(tasks) == 1 and all(np.all(s.w[:, 0] > 0) for s in splits)
        result = [(np.array([fingerprint(str(smi)) for smi in s.ids]),
                   s.y[:, 0].copy(), s.ids.copy()) for s in splits]
    print(tasks[0], "train / valid / test:", *(len(y) for _, y, _ in result))
    return result

from rdkit.Chem import MACCSkeys

def fingerprint(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        raise ValueError(smiles)
    return np.asarray(MACCSkeys.GenMACCSKeys(mol), dtype=float)


In [ ]:
%%writefile -a "{script}"
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import ConstantKernel, RBF, WhiteKernel

(xt, yt, _), (xv, yv, _), (xs, ys, ids) = read_splits(dc.molnet.load_delaney)
kernel = ConstantKernel(1.0) * RBF(length_scale=3.0) + WhiteKernel(noise_level=0.1)
model = GaussianProcessRegressor(kernel=kernel, alpha=1e-8, normalize_y=True,
                                 random_state=SEED)
model.fit(xt, yt)
print("fitted kernel:", model.kernel_)
mu, sd = model.predict(xs, return_std=True)
covered = np.abs(ys - mu) <= 1.96 * sd
print("test MAE", round(mean_absolute_error(ys, mu), 4), "covered", sum(covered), "/", len(ys))
print("first", ids[0], "observed / mean / SD", np.round([ys[0], mu[0], sd[0]], 4))
# 依原資料順序取前十二筆，再依預測均值排列以便讀圖。
idx = np.argsort(mu[:12])
fig, ax = plt.subplots(figsize=(7.3, 4.2), layout="constrained")
ax.errorbar(np.arange(12), mu[idx], yerr=1.96 * sd[idx], fmt="o", color="#196b74",
            capsize=3, label="Predictive mean and 95% interval")
ax.scatter(np.arange(12), ys[idx], marker="x", color="#b34d26", label="Measured")
ax.set(xticks=np.arange(12), xticklabels=idx, xlabel="Test row (first 12)",
       ylabel="Solubility, log S")
ax.legend()
fig.savefig(figure_dir / "09-10-gaussian-processes.png", dpi=160)


In [ ]:
from IPython.display import Image, display

subprocess.run([python, str(script)], check=True)
figure_dir = Path("../../figures") if Path("../../outline.yml").is_file() else Path("figures")
display(Image(filename=str(figure_dir / "09-10-gaussian-processes.png")))